# Persistence — Interview Notes

Persistence is the core feature of LangGraph using which the state of the entire workflow is stored at a specific location (memory, SQLite, Postgres, etc.).

Imp Pointer → It can store the final as well as intermediate state values with the help of checkpointers.

__What is Checkpointing?__

Checkpointing is the process of saving a snapshot of the graph's state at a point in time.

In simple words: every time something changes in the graph, LangGraph takes a "photo" of the current state and saves it. This saved photo is called a checkpoint.

A checkpoint stores:

The current state values (all the data at that point)

The next node(s) that should run

Metadata — like which step number this is, and what caused this update

How Checkpointing Works — Super-Steps

LangGraph doesn't save the state after every tiny change. It saves state after every __super-step__.

What is a super-step? 

A super-step = a collection of a single / multiple nodes that forms a part of graph.

If only one node runs at a time → 1 node = 1 super-step.

If multiple nodes run in parallel (like in a Parallelization workflow) → all of them together = 1 super-step.

Simple flow:

Super-step 1 runs → state updates → checkpoint saved

Super-step 2 runs → state updates → checkpoint saved

Super-step 3 runs → state updates → checkpoint saved
...

So basically: run → update state → save checkpoint → repeat, until the graph reaches END.

This gives us a full history of checkpoints — one for every super-step — not just the final result.

__Threads (Very Important Concept)__

A thread is like a unique ID for one specific run/conversation of the graph.

Every time you invoke a graph with a thread_id, all checkpoints from that run get saved under that thread_id.

Different thread_id = completely separate, independent state/history.

Same thread_id = LangGraph will load the last saved checkpoint for that thread and continue from there.

__Why threads matter:__

They let one application handle multiple users/conversations at the same time without mixing up their states.

Example: user A's chat and user B's chat can run on the same graph but stay completely isolated, because they use different thread_ids.

__python__

config = {"configurable": {"thread_id": "1"}}

graph.invoke({"messages": [...]}, config)

If you call invoke again with the same thread_id, it picks up right where it left off (because it loads the last checkpoint of that thread).

__How to View the History of Changes__

LangGraph gives you a method to see every checkpoint that was ever saved for a thread:

__python__

list(graph.get_state_history(config))

This returns a list of all checkpoints (from the very first super-step to the latest), each containing:

The state at that point

The next node to run

Metadata (step number, etc.)

__To get just the latest/current state (not the full history):__

python

graph.get_state(config)

Each checkpoint also has a __checkpoint_id__, which uniquely identifies that exact version. 

You can pass this checkpoint_id back into the config to jump to (replay from) that exact past checkpoint — this is what makes Time Travel possible.

4 Advantages of Persistence (using Checkpointers)

1. Short-Term Memory

Since every super-step's state is saved under a thread_id, the graph remembers everything that happened earlier in that same thread — like remembering earlier messages in a conversation.

2. Time Travel

Because every super-step is checkpointed (not just the final one), you can go back to any earlier checkpoint using its checkpoint_id and:

Replay from that point, or
Fork into a new/alternate path from that point

This is useful for debugging — you can see exactly what the state looked like at each step.

3. Human-in-the-Loop (HITL)

Since state is saved after every super-step, the graph can be paused at a specific node, wait for a human to review/approve/edit something, and then resume from that exact saved checkpoint — instead of losing everything or starting over.

4. Fault Tolerant (Most Important)

If a node fails midway, you don't lose the whole run — the graph can resume from the last successfully saved checkpoint, instead of starting from START again. Persistence is what makes this recovery possible.

Quick Interview Summary

"LangGraph saves the graph's state after every super-step using checkpointers — this is called checkpointing. These checkpoints are grouped under a thread_id, which is what allows multiple independent conversations to run on the same graph without mixing up. Because we keep a full history of checkpoints (viewable with get_state_history), and not just the final result, we get four benefits for free: short-term memory, time travel, human-in-the-loop, and fault tolerance."

In [24]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [25]:
load_dotenv()

True

In [26]:
llm = ChatOpenAI()

In [27]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [28]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [29]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [30]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [31]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': "Why did the pizza go to the therapist? Because it had too many toppings and couldn't handle the pressure!",
 'explanation': 'This joke plays on the idea of a pizza having "too many toppings" as a metaphor for feeling overwhelmed or stressed. By saying that the pizza went to a therapist because it couldn\'t handle the pressure, it humorously suggests that even inanimate objects like pizza can experience emotional problems. It also plays on the common theme of people seeking therapy to handle stress and anxiety. Overall, the joke uses wordplay and a clever twist to create a humorous scenario.'}

In [32]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist? Because it had too many toppings and couldn't handle the pressure!", 'explanation': 'This joke plays on the idea of a pizza having "too many toppings" as a metaphor for feeling overwhelmed or stressed. By saying that the pizza went to a therapist because it couldn\'t handle the pressure, it humorously suggests that even inanimate objects like pizza can experience emotional problems. It also plays on the common theme of people seeking therapy to handle stress and anxiety. Overall, the joke uses wordplay and a clever twist to create a humorous scenario.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199725-acc0-632c-8002-f01a28829280'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-16T12:59:45.383670+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199725-98ae-618e-8001-1ca7

In [33]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist? Because it had too many toppings and couldn't handle the pressure!", 'explanation': 'This joke plays on the idea of a pizza having "too many toppings" as a metaphor for feeling overwhelmed or stressed. By saying that the pizza went to a therapist because it couldn\'t handle the pressure, it humorously suggests that even inanimate objects like pizza can experience emotional problems. It also plays on the common theme of people seeking therapy to handle stress and anxiety. Overall, the joke uses wordplay and a clever twist to create a humorous scenario.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199725-acc0-632c-8002-f01a28829280'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-16T12:59:45.383670+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199725-98ae-618e-8001-1ca

In [34]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': "Why did the pasta go to the party alone? Because it couldn't find a good date!",
 'explanation': 'This joke plays on the dual meaning of the word "date." In the context of the joke, "date" is used to refer to a romantic partner, but it is also a type of fruit commonly used in Middle Eastern and Mediterranean cuisine. The punchline suggests that the pasta couldn\'t find a good "date," meaning it couldn\'t find a suitable romantic partner to accompany it to the party, but also implying that it couldn\'t find a good actual date, like the fruit, to include in the dish. This clever play on words adds humor to the joke.'}

In [35]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta go to the party alone? Because it couldn't find a good date!", 'explanation': 'This joke plays on the dual meaning of the word "date." In the context of the joke, "date" is used to refer to a romantic partner, but it is also a type of fruit commonly used in Middle Eastern and Mediterranean cuisine. The punchline suggests that the pasta couldn\'t find a good "date," meaning it couldn\'t find a suitable romantic partner to accompany it to the party, but also implying that it couldn\'t find a good actual date, like the fruit, to include in the dish. This clever play on words adds humor to the joke.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f199725-ca81-6ec9-8002-1200c77b1fdb'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-16T12:59:48.503897+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f19

In [36]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta go to the party alone? Because it couldn't find a good date!", 'explanation': 'This joke plays on the dual meaning of the word "date." In the context of the joke, "date" is used to refer to a romantic partner, but it is also a type of fruit commonly used in Middle Eastern and Mediterranean cuisine. The punchline suggests that the pasta couldn\'t find a good "date," meaning it couldn\'t find a suitable romantic partner to accompany it to the party, but also implying that it couldn\'t find a good actual date, like the fruit, to include in the dish. This clever play on words adds humor to the joke.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f199725-ca81-6ec9-8002-1200c77b1fdb'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-16T12:59:48.503897+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1

# Time Travel Demo

In [37]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist? Because it had too many toppings and couldn't handle the pressure!", 'explanation': 'This joke plays on the idea of a pizza having "too many toppings" as a metaphor for feeling overwhelmed or stressed. By saying that the pizza went to a therapist because it couldn\'t handle the pressure, it humorously suggests that even inanimate objects like pizza can experience emotional problems. It also plays on the common theme of people seeking therapy to handle stress and anxiety. Overall, the joke uses wordplay and a clever twist to create a humorous scenario.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199725-acc0-632c-8002-f01a28829280'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-16T12:59:45.383670+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199725-98ae-618e-8001-1ca

In [38]:
workflow.invoke({'topic':'pasta'}, config=config1)

{'topic': 'pasta',
 'joke': "Why did the spaghetti go to the party? Because it's a-pasta-tively fabulous time!",
 'explanation': 'This joke plays on the word "pasta-tively" which is a play on the word "positively." The spaghetti went to the party because it is having a fabulous time and is "pasta-tively" enjoying itself. It\'s a punny way to make a simple joke about pasta going to a party and having a good time.'}

In [39]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the spaghetti go to the party? Because it's a-pasta-tively fabulous time!", 'explanation': 'This joke plays on the word "pasta-tively" which is a play on the word "positively." The spaghetti went to the party because it is having a fabulous time and is "pasta-tively" enjoying itself. It\'s a punny way to make a simple joke about pasta going to a party and having a good time.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199726-526e-6a94-8006-042f69c7d264'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-08-16T13:00:02.756651+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199726-4800-6c17-8005-4f2ed310c8b5'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the spaghetti go to the party? Because it's a-pasta-tively fabulous time!", 'explanation': 'This joke plays on th

In [42]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f199725-cb4b-6afa-8004-0798240947c6"}})

StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pizza go to the therapist? Because it had too many toppings and couldn't handle the pressure!", 'explanation': 'This joke plays on the idea of a pizza having "too many toppings" as a metaphor for feeling overwhelmed or stressed. By saying that the pizza went to a therapist because it couldn\'t handle the pressure, it humorously suggests that even inanimate objects like pizza can experience emotional problems. It also plays on the common theme of people seeking therapy to handle stress and anxiety. Overall, the joke uses wordplay and a clever twist to create a humorous scenario.'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f199725-cb4b-6afa-8004-0798240947c6'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-08-16T12:59:48.586554+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199725-cb48-6d64-8003-d6ae16f55

# Updating State

In [43]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f199725-cb4b-6afa-8004-0798240947c6", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f19972b-a872-644f-8005-bcd078e03c07'}}

In [45]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': "Why did the pizza go to the therapist? Because it had too many toppings and couldn't handle the pressure!", 'explanation': 'This joke plays on the idea of a pizza having "too many toppings" as a metaphor for feeling overwhelmed or stressed. By saying that the pizza went to a therapist because it couldn\'t handle the pressure, it humorously suggests that even inanimate objects like pizza can experience emotional problems. It also plays on the common theme of people seeking therapy to handle stress and anxiety. Overall, the joke uses wordplay and a clever twist to create a humorous scenario.'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19972b-a872-644f-8005-bcd078e03c07'}}, metadata={'source': 'update', 'step': 5, 'parents': {}}, created_at='2026-08-16T13:02:25.993610+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199725

In [46]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f19972b-a872-644f-8005-bcd078e03c07"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to school?\n\nTo get a little filling in!',
 'explanation': 'This joke plays on the double meaning of the word "filling." In the context of food, filling refers to the mixture of ingredients inside the samosa. However, in the context of education, filling can also refer to the knowledge or information that one gains while attending school. So, the joke suggests that the samosa went to school to literally get some filling (food) in it, as well as to figuratively gain some knowledge or education.'}

In [47]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to school?\n\nTo get a little filling in!', 'explanation': 'This joke plays on the double meaning of the word "filling." In the context of food, filling refers to the mixture of ingredients inside the samosa. However, in the context of education, filling can also refer to the knowledge or information that one gains while attending school. So, the joke suggests that the samosa went to school to literally get some filling (food) in it, as well as to figuratively gain some knowledge or education.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199730-d779-6159-8007-69323c8716c9'}}, metadata={'source': 'loop', 'step': 7, 'parents': {}}, created_at='2026-08-16T13:04:45.142425+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f199730-ce6e-67e6-8006-d578fd99366e'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'sa